## 볼린저 밴드 투자 전략 백테스팅
- 이동 평균선 생성 : 데이터의 20개의 평균 값
- 상단 밴드 생성 : 이동평균선 + (2 * 20개의 데이터의 표준편차 )
- 하단 밴드 생성 : 이동평균선 - (2 * 20개의 데이터의 표준편차 )
- 가격이 하단 밴드보다 낮은 경우 매수 
- 가격이 상단 밴드보다 높은 경우 매도

In [28]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from datetime import datetime

In [29]:
# AMZN(아마존) 데이터를 로드 
df = pd.read_csv("../csv/AMZN.csv", index_col = 'Date')
df.head()

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1997-05-15,2.437500,2.500000,1.927083,1.958333,1.958333,72156000
1997-05-16,1.968750,1.979167,1.708333,1.729167,1.729167,14700000
1997-05-19,1.760417,1.770833,1.625000,1.708333,1.708333,6106800
1997-05-20,1.729167,1.750000,1.635417,1.635417,1.635417,5467200
1997-05-21,1.635417,1.645833,1.375000,1.427083,1.427083,18853200


In [30]:
# 결측치 데이터가 존재하는가?
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5563 entries, 1997-05-15 to 2019-06-24
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Open       5563 non-null   float64
 1   High       5563 non-null   float64
 2   Low        5563 non-null   float64
 3   Close      5563 non-null   float64
 4   Adj Close  5563 non-null   float64
 5   Volume     5563 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 304.2+ KB


In [31]:
# 결측치나 무한대 데이터를 제거 
flag = df.isin( [np.nan, np.inf, -np.inf] ).any(axis=1)

In [32]:
# flag의 부정을 해서 데이터를 필터링 
df = df.loc[~flag, ]

In [33]:
len(df)

5563

In [34]:
# 종가를 제외한 데이터를 제외 
df = df[['Adj Close']]
# 이동평균선 -> rolling()
df['center'] = df['Adj Close'].rolling(20).mean()


In [35]:
df.iloc[18:24, ]

,Adj Close,center
Date,,
1997-06-11,1.541667,NaN
1997-06-12,1.604167,1.574740
1997-06-13,1.583333,1.555990
1997-06-16,1.572917,1.548177
1997-06-17,1.505208,1.538021
1997-06-18,1.510417,1.531771


In [36]:
# 상단 밴드, 하단 밴드 생성 
std_value = 2 * df['Adj Close'].rolling(20).std()

df['ub'] = df['center'] + std_value
df['lb'] = df['center'] - std_value

In [37]:
df.iloc[19:25]

,Adj Close,center,ub,lb
Date,,,,
1997-06-12,1.604167,1.574740,1.836333,1.313146
1997-06-13,1.583333,1.555990,1.745696,1.366283
1997-06-16,1.572917,1.548177,1.719869,1.376485
1997-06-17,1.505208,1.538021,1.693045,1.382996
1997-06-18,1.510417,1.531771,1.680201,1.383341
1997-06-19,1.510417,1.535938,1.676462,1.395413


In [38]:
# index의 값을 시계열로 변경 
df.index = pd.to_datetime( df.index )

In [39]:
df.tail()

,Adj Close,center,ub,lb
Date,,,,
2019-06-18,1901.369995,1821.456500,1935.384678,1707.528322
2019-06-19,1908.790039,1824.020001,1943.535145,1704.504858
2019-06-20,1918.189941,1826.945495,1952.830613,1701.060378
2019-06-21,1911.300049,1831.736499,1962.964470,1700.508528
2019-06-24,1907.953857,1835.970190,1971.444249,1700.496132


In [40]:
# 투자 시작 시간 설정 
start = '2010-01-01'

test_df = df.loc[ start: , ]
test_df.head()

,Adj Close,center,ub,lb
Date,,,,
2010-01-04,133.899994,133.984001,141.460445,126.507556
2010-01-05,134.690002,133.839500,141.132776,126.546225
2010-01-06,132.250000,133.741500,141.066419,126.416581
2010-01-07,130.000000,133.536000,141.045671,126.026329
2010-01-08,133.520004,133.646500,141.082939,126.210062


In [41]:
# 구매 상태를 입력할수 있는 공간 생성 
test_df['trade'] = ''

C:\Users\ekfla\AppData\Local\Temp\ipykernel_12480\3747825671.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['trade'] = ''


#### 보유 내역 추가 
- 조건식 
    - 상단 밴드보다 수정 주가가 높거나 같은 경우
        - 내가 현재 보유중이라면? --> 전날짜의 trade가 'buy'라면 
            - 매도 (trade = '')
        - 보유중이 아니라면?
            - 유지 (trade = '')
    - 하단 밴드보다 수정 주가가 작거나 같은 경우 
        - 내가 보유중이라면?
            - 유지 (trade = 'buy')
        - 보유중이 아니라면?
            - 매수 (trade = 'buy')
    - 수정 주가가 밴드 사이에 존재하는 경우 
        - 내가 보유중이라면
            - 유지 (trade = 'buy')
        - 보유중이 아니라면
            - 유지 (trade = '')

In [46]:
for i in test_df.index:
    # print(i)
    # break
    if test_df.loc[i, 'Adj Close'] >= test_df.loc[i, 'ub']:
        # 상단밴드보다 수정 주가가 높은 경우 
        # 보유중이라면 trade = '', 보유중이 아니라면 trade =''
        test_df.loc[i, 'trade'] = ''
    elif test_df.loc[i, 'Adj Close'] <= test_df.loc[i, 'lb']:
        # 하단밴드보다 수정 주가가 낮은 경우 
        # 보유중인 경우 trade = 'buy' 보유중이 아닌 경우 trade = 'buy'
        test_df.loc[i, 'trade'] = 'buy'
    else:
        # 두 상황 모두 유지 -> 보유중인 경우 trade = 'buy' 보유중이 아니면 trade = ''
        # 전날의 trade가 buy인 경우 : 보유중
        if test_df.shift().loc[i, 'trade'] == 'buy':
            test_df.loc[i, 'trade'] = 'buy'
        else:
            test_df.loc[i, 'trade'] = ''
        # test_df.loc[i,'trade'] = test_df.shift().loc[i, 'trade']

In [47]:
test_df['trade'].value_counts()

trade
       1521
buy     863
Name: count, dtype: int64

### 수익율 계산
- 매수한 날의 수정주가 와 매도한 날의 수정 주가를 이용하여 수익율 계산
- 매수한 날의 수정 주가
    - 전날의 trade = ''이고 오늘의 trade = 'buy' 인 날짜의 수정 주가 
- 매도한 날의 수정 주가
    - 전날의 trade = 'buy'이고 오늘의 trade = ''인 날짜의 수정 주가 
- 수익율 
    - 매도날의 수정주가 / 매수날의 수정주가

In [48]:
# 수익율 컬럼을 생성 기본값은 1로 채워준다. 
test_df['rtn'] = 1

C:\Users\ekfla\AppData\Local\Temp\ipykernel_12480\2383034271.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['rtn'] = 1


In [49]:
test_df.head()

,Adj Close,center,ub,lb,trade,rtn
Date,,,,,,
2010-01-04,133.899994,133.984001,141.460445,126.507556,,1
2010-01-05,134.690002,133.839500,141.132776,126.546225,,1
2010-01-06,132.250000,133.741500,141.066419,126.416581,,1
2010-01-07,130.000000,133.536000,141.045671,126.026329,,1
2010-01-08,133.520004,133.646500,141.082939,126.210062,,1


In [ ]:
for i in test_df.index:
    # 매수 가격을 형성 
    if (test_df.shift().loc[i, 'trade'] == '') & (test_df.loc[i, 'trade'] == 'buy'):
        buy = test_df.loc[i, 'Adj Close']
        print(f"매수일 : {i}, 매수가 : {buy}")
    # 매도 가겨을 형성 
    elif (test_df.shift().loc[i, 'trade'] == 'buy') & (test_df.loc[i, 'trade'] == ''):
        sell = test_df.loc[i, 'Adj Close']
        # 수익율 계산 
        rtn = sell / buy
        test_df.loc[i, 'rtn'] = rtn
        print(f"매도일 : {i}, 매도가 : {sell}, 수익율 : {rtn}")

In [52]:
# 누적 수익율 계산 -> 9년 정도의 데이터에서 rtn의 값을 모두 누적 곱
acc_rtn = 1

for i in test_df.index:
    rtn = test_df.loc[i, 'rtn']
    acc_rtn *= rtn
acc_rtn

np.float64(3.138061358619031)

In [53]:
# 누적 곱 이라는 함수 cumprod()
test_df['acc_rtn'] = test_df['rtn'].cumprod()

C:\Users\ekfla\AppData\Local\Temp\ipykernel_12480\1630459153.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['acc_rtn'] = test_df['rtn'].cumprod()


In [54]:
test_df.iloc[-1, -1]

np.float64(3.138061358619031)

In [55]:
# buyandhold --> 
# 투자 기간 첫날 -> 매수가
# 투자의 마지막 날 -> 매도가
bnh_rtn = test_df.iloc[-1, 0] / test_df.iloc[0, 0]
print(bnh_rtn)

14.249095911087196


#### 볼린저 밴드 함수화 
1. 밴드를 생성하는 함수 
    - 매개변수
        - 데이터, 기준이 되는 컬럼(Adj Close), 투자의 시작 시간(2010-01-01), 종료시간(현재 시간), 묶음 데이터 개수(20)
        - _df, _col, _start, _end, _cnt
    - _df를 깊은 복사 
    - 인덱스가 Date가 아니라면 인덱스를 Date로 변경 
        - 컬럼들 중 이름이 Date가 존재한다면? -> index가 Date가 아니다
    - 인덱스를 시계열 변환
    - 시계열 데이터에서 time_zone을 제거 
    - 결측치, 무한대 데이터를 제외 
    - 기준이 되는 컬럼을 제외하고는 모두 제거 
    - 이동 평균선, 상단밴드, 하단밴드 생성 (파생변수 생성)
    - 시작시간과 종료 시간으로 데이터를 필터링 
    - 위에 결과를 되돌려준다. 

In [60]:
def create_band(
        _df, 
        _col = 'Adj Close', 
        _start = '2010-01-01', 
        _end = datetime.now(), 
        _cnt = 20
):
    # 복사본 생성 : 깊은 복사 
    df = _df.copy()
    # 인덱스가 Date인가? -> 컬럼중에 Date가 존재하는가?
    if 'Date' in df.columns:
        # 포함되어있다면? -> Date를 index로 변경
        df.set_index('Date', inplace=True)
    # index를 시계열로 변경 
    df.index = pd.to_datetime(df.index)
    # timezone 제거 
    df.index = df.index.tz_localize(None)
    # 결측치, 무한대 데이터를 제거 
    flag = df.isin([np.nan, np.inf, -np.inf]).any(axis=1)
    df = df.loc[~flag, ]
    # 기준이 되는 컬럼을 제외하고 모두 제거 
    df = df[[_col]]
    # 이동평균선, 상단밴드, 하단밴드 생성
    df['center'] = df[_col].rolling(_cnt).mean()
    std_value = 2 * df[_col].rolling(_cnt).std()
    df['ub'] = df['center'] + std_value
    df['lb'] = df['center'] - std_value
    # 시작 시간과 종료시간으로 필터링 
    df = df.loc[_start:_end, ]
    return df

In [72]:
df2 = pd.read_csv("../csv/aapl.csv")

band_df = create_band(df2)
band_df.head(1)

,Adj Close,center,ub,lb
Date,,,,
2010-01-04,26.782711,25.037723,27.046734,23.028713


2. 보유내역을 생성하는 함수 
    - 매개변수
        - 밴드가 생성된 데이터프레임
        - 기준이 되는 컬럼 (선택적)
    - trade 컬럼을 생성하여 '' 값으로 채워준다. 
    - 밴드의 값들과 기준이 되는 컬럼의 값을 이용하여 보유 내역을 생성
    - 결과는 되돌려준다. 

In [73]:
def create_trade(_df):
    # 기준이 되는 컬럼의 이름을 어떻게 알것인가? -> 첫함수의 결과값을 생각. -> columns => ['Adj Close', 'center', 'ub', 'lb']
    # 기준이 되는 컬럼은 _df에서 첫번쨰 컬럼의 이름이구나 .
    col = _df.columns[0]

    df = _df.copy()

    df['trade'] = ''

    for i in df.index:
        if df.loc[i, col] >= df.loc[i, 'ub']:
            # 매도 
            df.loc[i, 'trade'] = ''
        elif df.loc[i, col] <= df.loc[i, 'lb']:
            # 매수
            df.loc[i, 'trade'] = 'buy'
        else:
            if df.shift().loc[i, 'trade'] == 'buy':
                df.loc[i, 'trade'] = 'buy'
            else:
                df.loc[i, 'trade'] = ''
    return df

In [74]:
trade_df = create_trade(band_df)

trade_df['trade'].value_counts()

trade
       1439
buy     945
Name: count, dtype: int64

3. 수익율 계산 함수 
    - 매개변수 
        - 두번째 함수의 결과를 받아오는 데이터 매개변수 
    - 기준이 되는 컬럼의 이름을 변수에 저장
    - 데이터프테임에 rtn 컬럼을 생성하여 1로 채워준다.
    - 매수날과 매도날을 조건식을 생성하여 가격을 생성하고 수익율 계산 뒤 rtn 컬럼에 대입 
    - 누적 수익율 컬럼을 생성하여 누적수익율을 대입 
    - 생성된 데이터프레임과 최종 누적 수익율을 되돌려준다. 

In [75]:
def create_rtn(_df):
    col = _df.columns[0]
    df = _df.copy()

    df['rtn'] = 1

    # 수익율 계산
    for i in df.index:
        # 매수 
        if (df.shift().loc[i, 'trade'] == "") & (df.loc[i, 'trade'] == "buy"):
            buy = df.loc[i, col]
            print(f"매수일 : {i}, 매수가 : {buy}")
        elif (df.shift().loc[i, 'trade'] == "buy") & (df.loc[i, 'trade'] == '') :
            sell = df.loc[i, col]
            rtn  = sell / buy
            df.loc[i, 'rtn'] = rtn

            print(f"매도일 : {i}, 매도가 : {sell}, 수익율 : {rtn}")
    # 누적 수익율 계산
    df['acc_rtn'] = df['rtn'].cumprod()
    # 최종 수익율
    acc_rtn = df.iloc[-1, -1]

    return df, acc_rtn

In [ ]:
rtn_df, acc_rtn = create_rtn(trade_df)

In [77]:
acc_rtn

np.float64(1.3923287814461949)

In [79]:
rtn_df.iloc[-1, 0] / rtn_df.iloc[0, 0]

np.float64(7.436513727083117)

In [ ]:
class Investing():
    # 생성자 함수 
    # 객체에서 사용할 변수들은 저장하는 기능
    # 투자 전략에서 필요한 변수들 ( 데이터, 기준이 되는 컬럼, 투자의 시작시간, 종료시간 )
    def __init__(self, _df, _col = 'Adj Close', _start = '2010-01-01', _end = datetime.now()):
        self.df = _df
        self.col = _col
        self.start = _start
        self.end = _end
    # 바이앤홀드 함수 
    def bnh(self):
        # 수익율 계산 
        df = self.df.copy()
        if 'Date' in df.columns:
            df.set_index('Date', inplace=True)
        df.index = pd.to_datetime(df.index)
        # display(df.head())
        df = df.loc[self.start : self.end, [self.col]]
        buy = df.iloc[0, 0]
        sell = df.iloc[-1, 0]
        return sell / buy
    # 볼린져 밴드 함수 
    def boll(self, _cnt = 20):
        band_df = create_band(self.df, self.col, self.start, self.end, _cnt)
        trade_df = create_trade(band_df)
        rtn_df, acc_rtn  = create_rtn(trade_df)
        return rtn_df, acc_rtn

In [100]:
df3 = pd.read_csv("../csv/MSFT.csv")

In [101]:
invest = Investing(df3)

In [102]:
invest.bnh()

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1986-03-13,0.088542,0.101563,0.088542,0.097222,0.069996,1031788800
1986-03-14,0.097222,0.102431,0.097222,0.100694,0.072496,308160000
1986-03-17,0.100694,0.103299,0.100694,0.102431,0.073746,133171200
1986-03-18,0.102431,0.103299,0.098958,0.099826,0.071871,67766400
1986-03-19,0.099826,0.100694,0.097222,0.098090,0.070621,47894400


np.float64(5.6387313298309785)